# 0. Install Dependencies

In [14]:
# !pip install tensorflow==2.3.0
# !pip install gym
# !pip install keras
# !pip install keras-rl2

# 1. Test Random Environment with OpenAI Gym

In [15]:
from gym import Env
from gym.spaces import Discrete, Box
import numpy as np
import random
import mujoco_py
import os
import math

In [16]:
class SnakeEnv(Env):
    def __init__(self):
        # Define snake model path
        self.snake_model_path = "Model/simplified-snake.xml"
        # Load snake model
        self.snake_model = mujoco_py.load_model_from_path(self.snake_model_path)
        # Load simulation engine
        self.sim = mujoco_py.MjSim(self.snake_model)
        # Visualize simulation environment
        self.viewer = mujoco_py.MjViewer(self.sim)
        # Actions we can take, left, stay, right
        self.action_space = Discrete(3)
        # Distance array
        self.observation_space = Box(low=np.array([0]), high=np.array([100]))
        # Set goal pos
        self.goal = [6, 2.5]
        # Set start distance
        self.dist = (self.goal[0] - self.sim.data.qpos[6])**2 + (self.goal[1] - self.sim.data.qpos[7])**2
        self.previous_dist = self.dist
        self.sim_length = 250000

    def step(self, action1, action2):
        # Apply action
        self.previous_dist = self.dist
        self.sim.data.ctrl[0] = (action1-1) * 0.01
        self.sim.data.ctrl[1] = (action2-1) * 0.01
        self.sim.step()
        self.dist = (self.goal[0] - self.sim.data.qpos[6])**2 + (self.goal[1] - self.sim.data.qpos[7])**2
        # Reduce simulation length by 1 step
        self.sim_length -= 1

        # Calculate reward
        if self.dist < self.previous_dist:
            reward = 1
        else:
            reward = -1

        # Check if simulation is done
        if self.sim_length <= 0 or self.dist <= 2:
            done = True
        else:
            done = False

        # Set placeholder for info
        info = {}

        # Return step information
        return self.dist, reward, done, info

    def render(self):
        self.viewer.render()

    def reset(self):
        # Reset pos
        # Load simulation engine
        self.sim = mujoco_py.MjSim(self.snake_model)
        # Visualize simulation environment
        self.viewer = mujoco_py.MjViewer(self.sim)
        self.dist = (self.goal[0] - self.sim.data.qpos[6])**2 + (self.goal[1] - self.sim.data.qpos[7])**2
        self.previous_dist = self.dist
        self.sim_length = 250000

        return self.dist

In [17]:
env = SnakeEnv()

Creating window glfw


In [18]:
env.observation_space.sample()

array([66.88252], dtype=float32)

In [19]:
episodes = 2
for episode in range(1, episodes + 1):
    dist = env.reset()
    done = False
    score = 0

    while not done:
        action1 = env.action_space.sample()
        action2 = env.action_space.sample()
        n_dist, reward, done, info = env.step(action1, action2)
        #env.render()
        score += reward
    print('Episode:{} Score:{}'.format(episode, score))

Episode:1 Score:-2108
Creating window glfw
Episode:2 Score:326
Creating window glfw
Episode:1 Score:1910
Creating window glfw
Episode:2 Score:508
